In [1]:
from smac import Scenario, BlackBoxFacade

from predict_emotion import AudioAugmentation, return_device, EmotionDataset, EmotionModel, compute_loss, validate
from transformers import Wav2Vec2Processor, Wav2Vec2Config
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from ConfigSpace import Configuration, ConfigurationSpace, Float
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from tqdm import tqdm
import numpy as np


2025-01-25 14:53:12.734724: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-25 14:53:12.734764: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-25 14:53:12.735970: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-25 14:53:12.742395: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-25 14:53:13.675013: W tensorflow/comp

In [2]:
class EmotionModelCV:

    def __init__(self, train_dataloader, val_dataloader, device, pretrained_model):
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.device = device
        self.pretrained_model = pretrained_model


    @property
    def configspace(self) -> ConfigurationSpace:
        cs = ConfigurationSpace(seed = 0)
        alpha = Float("alpha", (0.0, 1.0), default=0.5)
        beta = Float("beta", (0.0, 1.0), default=0.5)
        cs.add([alpha, beta])
        return cs
    

    def train(self, config: Configuration, seed: int = 42) -> float:
        alpha = config["alpha"]
        beta = config["beta"]

        config_model = Wav2Vec2Config.from_pretrained(self.pretrained_model)
        

        model = EmotionModel(config_model).to(self.device)
        optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=1e-3)
        scheduler = OneCycleLR(optimizer, max_lr=1e-4, steps_per_epoch=len(self.train_dataloader), epochs=10)
        train_losses = []
        val_losses = []

        for epoch in range(5):
            model.train()
            for batch in tqdm(self.train_dataloader):

                optimizer.zero_grad()
            
                loss, _, _ = compute_loss(model, self.device, batch, alpha, beta)
                if loss is None: continue 
                train_losses.append(loss)
                loss.backward()

                optimizer.step()
                loss = loss.item()

            print(f"Training loss: {np.mean(train_losses)}")

            val_loss = validate(model, self.device, self.val_dataloader, alpha, beta)
            val_losses.append(val_loss)
            scheduler.step(val_loss)


        return np.mean(val_losses)

In [3]:
pretrained_model = "facebook/wav2vec2-base"
processor = Wav2Vec2Processor.from_pretrained(pretrained_model) 
muse = pd.read_pickle("../../data/MuSe_sample").sample(frac=1, random_state=42)
iemocap = pd.read_pickle("../../data/IEMOCAP_useful").sample(frac=1, random_state=42)

df = pd.concat([iemocap, muse]).sample(frac=1, random_state=42)

df.drop(columns = ["Name"], inplace = True)
print(df)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

augmenter = AudioAugmentation(sample_rate=16000)

train_dataset = EmotionDataset(train_df, processor, augmenter, False)
test_dataset = EmotionDataset(test_df, processor, augmenter, False)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True,\
                           num_workers=4, pin_memory=True, drop_last = True, )
val_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=True,\
                           num_workers=4, pin_memory=True, drop_last = True,)

device = return_device()


/home/lucab/Recommersion/myenv/lib/python3.10/site-packages/transformers/configuration_utils.py:311: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


       Valence   Arousal                                           wav_file
4755  0.839989  0.562950  [-0.01876831, -0.022216797, -0.023742676, -0.0...
1333  0.666667  0.625000  [0.015930176, 0.04156494, 0.064453125, 0.08090...
6810  0.222222  0.500000  [0.0036621094, 0.0029907227, 0.0026550293, 0.0...
2651  0.787327  0.471909  [-0.016723633, -0.018585205, -0.037475586, -0....
2412  0.096797  0.510337  [-0.026885986, -0.017242432, -0.004486084, -0....
...        ...       ...                                                ...
3955  0.711117  0.670676  [-0.02130127, -0.025054932, -0.02960205, -0.03...
6586  0.250290  0.840794  [-0.0024108887, -0.0015258789, -0.0002746582, ...
615   0.222222  0.750000  [0.02810669, 0.027008057, 0.019897461, 0.01052...
9156  0.111111  0.625000  [-0.039398193, -0.04046631, -0.031463623, -0.0...
4796  0.349781  0.556065  [0.0033874512, 0.002166748, 0.0013427734, 9.15...

[19172 rows x 3 columns]


In [5]:
# Configure the scenario for SMAC

model = EmotionModelCV(train_dataloader, val_dataloader, device, pretrained_model)
scenario = Scenario(
        model.configspace,
        n_trials=50,         
        deterministic=True
    )


smac = BlackBoxFacade(
        scenario = scenario,
        target_function = model.train,
        #overwrite=True,  
    )

best_config = smac.optimize()
print(f"Best configuration: {best_config}")

[INFO][abstract_initial_design.py:87] Reducing the number of initial configurations from 16 to 12 (max_ratio == 0.25).
[INFO][abstract_initial_design.py:139] Using 12 initial design configurations and 0 additional configurations.
[INFO][smbo.py:509] Continuing from previous run.
[INFO][abstract_intensifier.py:289] Added existing seed 209652396 from runhistory to the intensifier.
[INFO][abstract_intensifier.py:307] Using only one seed for deterministic scenario.


/home/lucab/Recommersion/myenv/lib/python3.10/site-packages/transformers/configuration_utils.py:311: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
 20%|█▉        | 187/958 [01:50<07:37,  1.69it/s]

[WARNING][abstract_runner.py:135] Target function returned infinity or nothing at all. Result is treated as CRASHED and cost is set to inf.
[WARNING][abstract_runner.py:141] Traceback: Traceback (most recent call last):
  File "/home/lucab/Recommersion/myenv/lib/python3.10/site-packages/smac/runner/target_function_runner.py", line 190, in run
    rval = self(config_copy, target_function, kwargs)
  File "/home/lucab/Recommersion/myenv/lib/python3.10/site-packages/smac/runner/target_function_runner.py", line 264, in __call__
    return algorithm(config, **algorithm_kwargs)
  File "/tmp/ipykernel_3183254/4201704002.py", line 41, in train
    loss.backward()
  File "/home/lucab/Recommersion/myenv/lib/python3.10/site-packages/torch/_tensor.py", line 581, in backward
    torch.autograd.backward(
  File "/home/lucab/Recommersion/myenv/lib/python3.10/site-packages/torch/autograd/__init__.py", line 347, in backward
    _engine_run_backward(
  File "/home/lucab/Recommersion/myenv/lib/python3.10/


 11%|█         | 101/958 [01:01<08:38,  1.65it/s]


KeyboardInterrupt: 